# 03 — Assistente clínico com LangChain e fluxo LangGraph

Demonstração do sistema montado nos PRs 05 a 08: o modelo fine-tuned servido por um wrapper
LangChain, o contexto do paciente lido do banco, os limites de atuação impostos por código e
a trilha de auditoria da própria sessão.

Rode antes, a partir da raiz do repositório, se o banco ainda não existir:

```bash
python -m src.database.seed
```

Os adapters já estão em `data/fine_tuned/adapters` — o `02_fine_tuning.ipynb` registra o
treino que os produziu. Este notebook **não** treina nada: ele exercita o sistema pronto.

## Sobre o output committado

Este notebook é entregue com o output visível, e output de notebook entra no histórico do
git para não sair mais. Por isso a célula 7 **não** lê `logs/audit.jsonl` direto e **não**
exibe os dois campos de texto livre da trilha (`query` e `response_preview`): ela projeta uma
allowlist de metadados e flags, que é o que responde as perguntas de auditoria. O
`scripts/check_notebook_output.py` reprova o commit se esses campos escaparem para cá.

Pelo mesmo motivo, as perguntas usam só os tokens `[PACIENTE_00N]`. O `anonymize` do projeto
é denylist ancorada em contexto: um nome digitado solto numa pergunta iria em claro para a
trilha e daí para este arquivo.

## Célula 1 — Setup

`load_dotenv` e nada mais. Nenhuma célula deste notebook imprime variável de ambiente: o
`.env` tem o token do HuggingFace, e output de notebook é irreversível.

O `SESSAO` é sorteado a cada execução de propósito. O `audit.jsonl` acumula tudo o que já
rodou na máquina, e a célula 7 precisa mostrar **esta** execução — não o que sobrou de
tentativas anteriores que ninguém revisou.

In [1]:
import sys
import uuid
from pathlib import Path

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

from dotenv import load_dotenv

load_dotenv(RAIZ / ".env")

from src.assistant.chain import MedicalAssistant
from src.assistant.retriever import PatientRetriever
from src.audit.audit_logger import AuditLogger
from src.graph.clinical_flow import run_clinical_flow
from src.llm.guardrails import RODAPE_VALIDACAO

SESSAO = f"demo-{uuid.uuid4().hex[:8]}"
print(f"sessão desta execução: {SESSAO}")

/Users/lanunes/Documents/Repos/tech-challenge-group-24/venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


sessão desta execução: demo-a5a6d0c0


## Célula 2 — Qual modelo está respondendo

`_identifying_params` é propriedade do `MedicalMLXLLM` (`src/llm/model.py`), não do
assistente, e chega aqui pelo `self.llm` do `MedicalAssistant`. É o que prova, dentro da
entrega, qual adapter respondeu as células seguintes — o repositório tem dois
(`adapters`, de 500 iterações, e `adapters_best`, de 200), e as métricas do relatório
técnico só valem para o que o `ADAPTER_PATH` aponta.

O caminho sai relativo à raiz, como o resto do projeto faz no que é exibido: o absoluto cola
o resultado à máquina de quem rodou.

In [2]:
assistant = MedicalAssistant.from_env()
assistant.preload()


def curto(caminho) -> str:
    try:
        return str(Path(caminho).relative_to(RAIZ))
    except ValueError:
        return str(caminho)


for chave, valor in assistant.llm._identifying_params.items():
    print(f"{chave:14} {curto(valor) if 'path' in chave else valor}")

Fetching 11 files:   0%|          | 0/11 [00:00<?, ?it/s]

Fetching 11 files: 100%|██████████| 11/11 [00:00<00:00, 4678.77it/s]

model_path     meta-llama/Llama-3.2-3B-Instruct
adapter_path   data/fine_tuned/adapters
revision       0cb88a4f764b7a12671c53f0838cd831a0843b95
max_tokens     512
temperature    0.7


## Célula 3 — Pergunta clínica sem paciente

Uma pergunta de protocolo, sem `patient_id`: o modelo responde do que aprendeu no
fine-tuning, sem contexto de prontuário.

O que observar é a **fonte**. O `SYSTEM_PROMPT` exige citação em toda resposta, e o
`chain.py` confere a fonte citada contra o contexto que o modelo recebeu: quando não
confere, ela é registrada como ausente (`source=None`) e sai um `UserWarning`. Sem
`patient_id` não há contexto de paciente contra o que conferir, então este é o caso em que a
verificação é mais estrita.

Não se afirma aqui que a fonte sai sempre citada, porque ela não sai — o relatório técnico
mede quanto, e a análise explica por quê.


### O que saiu nesta execução

O modelo citou `[Fonte: protocoloCID J45]` — sem o espaço, e **J45 é o CID da asma**, não o da
gastroenterite (A09). A conferência do `chain.py` rejeitou a fonte contra o contexto recebido,
gravou `source=None` / `tem_fonte=False` e emitiu o `UserWarning` acima. A resposta também
sugere eletrocardiograma para um quadro gastrointestinal.

Não é azar desta rodada. O relatório técnico mede a causa: dos 903 exemplos de treino, **10%**
citam fonte e **zero** respondem sobre dado estruturado de paciente, enquanto o prompt exige
citação em toda resposta. A explainability é cobrada na inferência e quase não foi treinada, e
o modelo preenche a lacuna recitando o protocolo que tiver mais à mão.

Compare com a célula 5, que faz a mesma coisa **com** contexto de paciente.

In [3]:
resposta = assistant.ask(
    "Quais são os sinais de alarme na gastroenterite aguda em adultos?",
    session_id=SESSAO,
)

print(resposta["response"])
print()
print(f"fonte que conferiu contra o contexto: {resposta['source']!r}")
print(f"guardrail acionado: {resposta['guardrail_triggered']}")

[Fonte: protocoloCID J45] Para gastroenterite aguda (CID J45), o exame de referência é avaliação clínica. Solicitar eletrocardiograma quando houver dúvida diagnóstica e documentar a decisão clínica.

 Sinais de alerta incluem hematomas gástricos, choque e septicemia.
[Requer validação médica por profissional habilitado]

fonte que conferiu contra o contexto: None
guardrail acionado: False


/var/folders/vx/h1dj5jxx0_s68yrtrs_8mx6w0000gp/T/ipykernel_58722/2931937533.py:1: UserWarning: Fonte citada sem correspondência no contexto do paciente: 'protocoloCID J45'. Registrada na trilha como ausente.
  resposta = assistant.ask(


## Célula 4 — Tentativa de prescrição: o limite de atuação

O requisito é "nunca prescrever diretamente, sem validação humana". Duas camadas o
implementam, e esta célula mostra as duas:

1. `check_prescription_attempt` roda **antes** da inferência e injeta o aviso no próprio
   contexto, para o modelo receber o limite junto do dado em vez de só levar o carimbo depois;
2. `validate_response` roda **depois** e garante o rodapé de validação mesmo que o modelo
   tenha ignorado a instrução.

A pergunta contém o radical `prescrev-` por necessidade, não por estilo. O
`check_prescription_attempt` casa radicais (`prescr`, `receit`, `administr`), e há registro
entre as limitações conhecidas do projeto de que ele **não** pega posologia escrita sem
radical — "500 mg de cetoprofeno de 8/8h" passa. Usar uma frase natural que fura a denylist
demonstraria o guardrail falhando dentro da própria entrega; o alcance da regra é discutido
no relatório técnico, onde é análise e não payload.


### O que saiu nesta execução

O guardrail acionou: `guardrail_triggered=True`, motivo `prescricao_na_pergunta` (visível na
célula 7), o `AVISO_PRESCRICAO` abrindo a resposta e o rodapé de validação fechando.

Nenhuma das duas marcas dependeu de o modelo cooperar — e foi bom, porque ele não cooperou:
ignorou a pergunta sobre ondansetrona e repetiu o texto de protocolo da célula 3. É a
propriedade que se pede de um limite de atuação: ele vale **também** quando a geração é ruim.
Se dependesse da obediência do modelo, seria uma sugestão, não um limite.

In [4]:
tentativa = assistant.ask(
    "Posso prescrever ondansetrona 8 mg para a náusea desse quadro?",
    session_id=SESSAO,
)

print(tentativa["response"])
print()
print(f"guardrail acionado: {tentativa['guardrail_triggered']}")
print(f"rodapé de validação no fim da resposta: {tentativa['response'].rstrip().endswith(RODAPE_VALIDACAO)}")

[Este assistente não emite prescrição. O conteúdo abaixo é apoio à decisão clínica e precisa de validação por profissional habilitado antes de qualquer conduta.]

[Fonte: protocolo J45] Para gastroenterite aguda (CID J45), o exame de referência é avaliação clínica. Solicitar eletrocardiograma quando houver dúvida diagnóstica e documentar a decisão clínica.
[Requer validação médica por profissional habilitado]

guardrail acionado: True
rodapé de validação no fim da resposta: True


/var/folders/vx/h1dj5jxx0_s68yrtrs_8mx6w0000gp/T/ipykernel_58722/73119381.py:1: UserWarning: Fonte citada sem correspondência no contexto do paciente: 'protocolo J45'. Registrada na trilha como ausente.
  tentativa = assistant.ask(


## Célula 5 — Pergunta contextualizada por paciente

O `patient_id` é confirmado antes contra o banco. O `ask` deixa `PacienteNaoEncontrado`
subir de propósito — responder sobre um paciente inexistente, sem contexto mas parecendo que
teve, é pior que falhar —, então um identificador errado aqui interromperia o notebook.

`[PACIENTE_005]` é diabético tipo 2, tem alergia a iodo e sulfa e exatamente um exame
pendente. As alergias importam: o `chain.py` confere a resposta contra a lista do prontuário
e carimba o alerta por código, sem depender de o modelo ter lembrado.


### O que saiu nesta execução

Aqui o sistema acerta, e o contraste é o ponto mais importante deste notebook.

Com o contexto injetado, o modelo respondeu `Exame pendente: hemoglobina glicada (CID E11).
Solicitado em 02/09/2026` — que é **exatamente** o que o prontuário registra, tipo e data,
conferível contra o `get_pending_exams` impresso logo acima. A fonte citada conferiu contra o
contexto e foi gravada na trilha (`tem_fonte=True`, interação 3 da célula 7).

Mesmo modelo, mesma execução, minutos de diferença: sem contexto (célula 3) ele recita o
protocolo errado; com contexto ele lê o dado daquele paciente. É o que a contextualização do
LangChain entrega, e é o que o fine-tuning sozinho não entregou.

`alergias_alertadas` sai vazio porque nem a pergunta nem a resposta mencionaram iodo ou sulfa.
O carimbo de alergia é acionado por citação da substância, não pela mera existência dela no
prontuário — alertar em toda resposta de um paciente alérgico treinaria quem lê a ignorar o
alerta.

In [5]:
retriever = PatientRetriever.from_env()
disponiveis = retriever.listar_pacientes()
print(f"{len(disponiveis)} pacientes no banco — os cinco primeiros: {disponiveis[:5]}")

PACIENTE = "[PACIENTE_005]"
assert PACIENTE in disponiveis, f"{PACIENTE} não está no banco"

pendentes = retriever.get_pending_exams(PACIENTE)
print(f"{PACIENTE}: {len(pendentes)} exame(s) pendente(s) no prontuário")
print()

contextual = assistant.ask(
    "Quais exames deste paciente ainda estão pendentes?",
    patient_id=PACIENTE,
    session_id=SESSAO,
)

print(contextual["response"])
print()
print(f"contexto do paciente injetado no prompt: {contextual['patient_context_used']}")
print(f"fonte que conferiu contra o contexto: {contextual['source']!r}")
print(f"alergias alertadas por código: {contextual['alergias_alertadas']}")

20 pacientes no banco — os cinco primeiros: ['[PACIENTE_001]', '[PACIENTE_002]', '[PACIENTE_003]', '[PACIENTE_004]', '[PACIENTE_005]']
[PACIENTE_005]: 1 exame(s) pendente(s) no prontuário



[Fonte: exames PENDENTES (aguardando realização ou resultado)] Exame pendente: hemoglobina glicada (CID E11). Solicitado em 02/09/2026.
[Requer validação médica por profissional habilitado]

contexto do paciente injetado no prompt: True
fonte que conferiu contra o contexto: 'exames PENDENTES (aguardando realização ou resultado)'
alergias alertadas por código: []


## Célula 6 — Fluxo automatizado: os dois ramos da condicional

O grafo tem cinco nós e uma aresta condicional:

```
intake → check_exams →  alert_team        → human_validation
                     ↘  suggest_treatment ↗
```

A decisão sai de `pending_exams`, que o `check_exams` acabou de ler do banco. Os dois
pacientes abaixo exercitam os dois lados — um só mostraria metade do grafo, e a condicional
é o entregável:

- `[PACIENTE_013]` tem 3 exames pendentes → `alert_team`, que **não chama o modelo**;
- `[PACIENTE_002]` não tem nenhum → `suggest_treatment`, que pede conduta pelo assistente
  completo (com contexto, guardrail e trilha), e não falando direto com o LLM.

As duas execuções recebem o `SESSAO` desta demonstração, então caem na mesma sessão das
células anteriores e saem juntas na célula 7.


### O que saiu nesta execução

Os dois ramos se comportaram como o desenho previa.

`[PACIENTE_013]`, com 3 exames pendentes, foi para `alert_team`: os três alertas saíram
formatados com tipo e data e **o modelo não foi chamado** — o ramo é determinístico, lê do
banco e não gera texto. `[PACIENTE_002]`, sem pendências, foi para `suggest_treatment`, e a
conduta citou `[Fonte: protocolo A09]`. A09 é o CID de gastroenterite, que é a condição
registrada para ele, e desta vez a fonte conferiu (`tem_fonte=True`, interação 5 da célula 7).

Os dois terminam com `requires_validation=True`: o `human_validation_node` é o nó de saída dos
dois caminhos, e não um passo do ramo que sugere conduta.

In [6]:
for paciente in ("[PACIENTE_013]", "[PACIENTE_002]"):
    estado = run_clinical_flow(paciente, session_id=SESSAO)
    caminho = "alert_team" if estado["pending_exams"] else "suggest_treatment"

    print(f"===== {paciente} =====")
    print(f"exames registrados: {len(estado['exams'])}  |  pendentes: {len(estado['pending_exams'])}")
    print(f"caminho após check_exams: {caminho}")
    print(f"requer validação humana: {estado['requires_validation']}")

    for alerta in estado["alerts"]:
        print(f"  {alerta}")
    if estado["suggestions"]:
        print(f"\nconduta sugerida:\n{estado['suggestions']}")
    print()

===== [PACIENTE_013] =====
exames registrados: 4  |  pendentes: 3
caminho após check_exams: alert_team
requer validação humana: True
  [ALERTA PARA A EQUIPE] O paciente [PACIENTE_013] tem 3 exame(s) pendente(s). Conduta terapêutica não foi sugerida por este fluxo enquanto houver resultado em aberto.
  Exame pendente: urocultura com antibiograma (solicitado em 31/08/2026)
  Exame pendente: perfil lipídico (solicitado em 30/08/2026)
  Exame pendente: hemograma completo (solicitado em 27/08/2026)



===== [PACIENTE_002] =====
exames registrados: 2  |  pendentes: 0
caminho após check_exams: suggest_treatment
requer validação humana: True

conduta sugerida:
[Fonte: protocolo A09] Avaliação: hidratação oral, dieta leve e sinais de alerta orientados. Achado esperado: melhora parcial dos sintomas. Solicitar avaliação clínica de hidratação para confirmação diagnóstica. Conduta: manter hidratação oral, dieta leve e sinais de alerta orientados. Solicitar reavaliação clínica em 24 horas.
[Requer validação médica por profissional habilitado]



## Célula 7 — Trilha de auditoria desta sessão

A trilha é consultada por `get_session_logs(SESSAO)` e **não** abrindo `logs/audit.jsonl`.
Duas razões: o arquivo acumula execuções anteriores que ninguém revisou, e ele é criado em
`0600` justamente para não ser lido por qualquer caminho.

Os campos exibidos são uma allowlist de metadados e flags — é o que responde as perguntas de
auditoria ("de qual paciente?", "houve guardrail e por quê?", "citou fonte que confere?").
Ficam de fora `query` e `response_preview`, os dois campos de texto livre derivados do
contexto clínico. `source` é a única exceção, porque é texto do modelo: ele entra porque já
passa pelo `anonimizar_fonte` e vira `None` quando não confere com o contexto.


### O que saiu nesta execução

As 5 interações são as das células 3 a 6, na ordem: duas sem paciente (pergunta de protocolo e
tentativa de prescrição), uma de `[PACIENTE_005]` e as duas do fluxo automatizado.

A interação 4 é a do `alert_team` e é a única com `tem_fonte=None`. O valor distingue dois
casos que um `False` juntaria: aquele ramo não chama o modelo, então não existe fonte a
conferir — diferente de "o modelo respondeu e não citou nada", que é o que as interações 1 e 2
registram.

Nenhum campo de texto livre (`query`, `response_preview`) aparece acima. Eles estão gravados no
`audit.jsonl`, que continua sendo a trilha completa para quem tem acesso ao arquivo; o que não
entra é o arquivo committado.

In [7]:
CAMPOS = (
    "timestamp",
    "session_id",
    "patient_id",
    "guardrail_triggered",
    "motivos",
    "tem_fonte",
    "source",
    "alergias_alertadas",
)

trilha = AuditLogger.from_env().get_session_logs(SESSAO)
print(f"{len(trilha)} interações registradas na sessão {SESSAO}\n")

for numero, entrada in enumerate(trilha, start=1):
    print(f"----- interação {numero} -----")
    for campo in CAMPOS:
        print(f"  {campo:20} {entrada.get(campo)!r}")
    print()

5 interações registradas na sessão demo-a5a6d0c0

----- interação 1 -----
  timestamp            '2026-09-12T22:34:31.714+00:00'
  session_id           'demo-a5a6d0c0'
  patient_id           None
  guardrail_triggered  False
  motivos              []
  tem_fonte            False
  source               None
  alergias_alertadas   []

----- interação 2 -----
  timestamp            '2026-09-12T22:35:01.055+00:00'
  session_id           'demo-a5a6d0c0'
  patient_id           None
  guardrail_triggered  True
  motivos              ['prescricao_na_pergunta']
  tem_fonte            False
  source               None
  alergias_alertadas   []

----- interação 3 -----
  timestamp            '2026-09-12T22:35:05.727+00:00'
  session_id           'demo-a5a6d0c0'
  patient_id           '[PACIENTE_005]'
  guardrail_triggered  False
  motivos              []
  tem_fonte            True
  source               'exames PENDENTES (aguardando realização ou resultado)'
  alergias_alertadas   []

----- inte

---

## O que este notebook mostra, e onde está a interpretação

Os quatro itens que a Fase 3 exige demonstrados estão aqui: o modelo personalizado
respondendo (células 2 e 3), o fluxo automatizado com a decisão condicional (célula 6), a
resposta contextualizada pelo prontuário (célula 5) e os logs com a validação das respostas
(células 4 e 7).

As células acima descrevem **o que saiu**. A interpretação — por que a citação de fonte falha
sem contexto, o que as métricas ROUGE-L e BLEU-4 medem e não medem, e quais limitações são
estruturais e quais são do dataset — está em [`docs/relatorio-tecnico.md`](../docs/relatorio-tecnico.md).